# Cybersecurity Risk Analysis

In [14]:
import pandas as pd
import statsmodels.api as sm

Here’s a detailed explanation of the methodology used in the program for performing the multivariate regression analysis:

**Data Loading and Preprocessing**

The first step involves reading the dataset and preparing it for regression analysis.

- **File Input**: The program reads a dataset from an Excel file (`file_path`) and a specific sheet (`sheet_name`) using `pandas.read_excel`.
- **Columns Selection**:
  - `stock_columns`: A list of stock columns representing the dependent variables. These are the series for which we perform separate regressions.
  - `index_columns`: A list of index columns representing the independent variables (predictors).


**Regression Analysis**

For each stock in `stock_columns`, the program performs a **multivariate ordinary least squares (OLS) regression** against the independent variables (indices in `index_columns`).

1. **Define the Variables**:
   - **Dependent Variable (`y`)**: The column corresponding to the current stock in the iteration.
   - **Independent Variables (`X`)**: The columns corresponding to the indices.
   - A constant term is added to the independent variables using `statsmodels.api.add_constant`. This constant term accounts for the intercept in the regression model.

2. **Model Fitting**:
   - The regression model is constructed using the OLS method (`statsmodels.api.OLS`) and then fitted using `.fit()`.
   - The OLS model estimates the relationship between the dependent variable (stock) and the independent variables (indices).

3. **Results Extraction**:
   - **Coefficients (`model.params`)**: The regression coefficients (including the intercept) are extracted. These coefficients represent the estimated effect of each independent variable on the dependent variable.
   - **P-values (`model.pvalues`)**: These are the significance levels of the coefficients. A small p-value (e.g., < 0.05) indicates that the corresponding coefficient is statistically significant.
   - **R-squared (`model.rsquared`)**: This value measures the proportion of variance in the dependent variable explained by the independent variables. Higher values indicate better model fit.


**Output Results**
The results of the regression are saved in a dictionary (`regression_results`) for each stock, structured as follows:
- **Coefficients**: A dictionary mapping each independent variable (including the intercept) to its coefficient.
- **P-values**: A dictionary mapping each independent variable to its p-value.
- **R-squared**: A single value representing the goodness of fit for the regression model.

**Iterative Analysis**
The program loops through each stock column, performing the regression independently for each stock. This ensures:
- Each stock's performance is modeled separately against the indices.
- The relationship between the stock and the indices is captured uniquely.

In [15]:
def perform_regressions(file_path, sheet_name, stock_columns, index_columns):
    """
    Effettua regressioni multivariate separatamente per ciascuna azione rispetto agli indici specificati.

    :param file_path: Percorso al file Excel contenente i dati.
    :param sheet_name: Nome del foglio nel file Excel.
    :param stock_columns: Lista delle colonne delle azioni.
    :param index_columns: Lista delle colonne degli indici.
    :return: Dizionario con i risultati delle regressioni per ogni azione.
    """
    # Carica i dati dal file Excel
    df = pd.read_excel(file_path, sheet_name=sheet_name)

    # Inizializza un dizionario per salvare i risultati delle regressioni
    regression_results = {}

    # Esegui una regressione separata per ogni azione rispetto ai due indici
    for stock in stock_columns:
        # Definisci le variabili indipendenti (gli indici) e dipendente (l'azione)
        X = df[index_columns]
        y = df[stock]

        # Aggiungi una costante per il modello
        X = sm.add_constant(X)

        # Esegui la regressione
        model = sm.OLS(y, X).fit()

        # Prepare significance information
        significance_info = {
            param: "Significant" if p < 0.05 else "Not significant"
            for param, p in model.pvalues.items()
        }

        # Save the results for this stock
        regression_results[stock] = {
            'coefficients': model.params.to_dict(),
            'p-values': model.pvalues.to_dict(),
            'significance': significance_info,
            'r-squared': model.rsquared
        }

    return regression_results

In [21]:
# Specifica i parametri
file_path     = './data-lp.xlsx'
sheet_name    = 'regressione'
stock_columns = [
    '@SPCB-YIELD', 'U:NOC-YIELD', '@MANH-YIELD', '@GEN-YIELD',
    '@ZIXI-YIELD', '@INTZ-YIELD', '@CHKP-YIELD', '@VRSN-YIELD',
    'U:NTIP-YIELD', '@TMICF-YIELD'
]
index_columns = ['NASDAQ 100 - YIELD', 'CYBER']

In [22]:
# Esegui le regressioni
results = perform_regressions(file_path, sheet_name, stock_columns, index_columns)

In [23]:
# Display results
for stock, result in results.items():
    print(f"Results for {stock}:")
    print("Coefficients:")
    print(result['coefficients'])
    print("P-values:")
    print(result['p-values'])
    print("Significance:")
    print(result['significance'])
    print(f"R-squared: {result['r-squared']}")
    print("-" * 50)

Results for @SPCB-YIELD:
Coefficients:
{'const': -0.5481471837093707, 'NASDAQ 100 - YIELD': 0.7436769464780226, 'CYBER': -0.2198510138047955}
P-values:
{'const': 0.17957423881047901, 'NASDAQ 100 - YIELD': 8.690834317465388e-05, 'CYBER': 0.9594983906469741}
Significance:
{'const': 'Not significant', 'NASDAQ 100 - YIELD': 'Significant', 'CYBER': 'Not significant'}
R-squared: 0.14087808639020472
--------------------------------------------------
Results for U:NOC-YIELD:
Coefficients:
{'const': -0.2877675589269533, 'NASDAQ 100 - YIELD': 0.6283592936750372, 'CYBER': 0.652117256069892}
P-values:
{'const': 0.08921155246496752, 'NASDAQ 100 - YIELD': 3.456473983218107e-13, 'CYBER': 0.7156934344721158}
Significance:
{'const': 'Not significant', 'NASDAQ 100 - YIELD': 'Significant', 'CYBER': 'Not significant'}
R-squared: 0.40801446614029635
--------------------------------------------------
Results for @MANH-YIELD:
Coefficients:
{'const': -0.2764037096265873, 'NASDAQ 100 - YIELD': 0.67999574108180

**Methodological Notes**

1. **Multivariate Regression**:
   - Each regression includes multiple independent variables (the two indices).
   - This allows the program to estimate the simultaneous effect of both indices on each stock.<p> </p>
   
2. **Significance Testing**:
   - The p-values are used to assess whether the relationship between the independent variables and the dependent variable is statistically significant.
   - If the p-value is small (e.g., < 0.05), the effect is considered statistically significant.

3. **Assumptions of OLS**:
   The program assumes the standard conditions of OLS regression:
   - Linearity: The relationship between the independent and dependent variables is linear.
   - Independence: Observations are independent of each other.
   - Homoscedasticity: The variance of the residuals is constant.
   - Normality: The residuals are normally distributed.